In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

folder = Path("/content/drive/MyDrive/Fotos Vacas Corral 6")

In [ ]:
import hashlib
import cv2
import numpy as np
import pandas as pd

from PIL import Image, ImageOps
from tqdm.auto import tqdm
%pip install -q pillow-heif

from pillow_heif import register_heif_opener

register_heif_opener()

# Find all files recursively under folder, but filter out any that have "por clasificar" in their parent folder parts
extensions = {".jpg", ".jpeg", ".heic"}

paths = sorted(
    p for p in folder.rglob("*")
    if p.is_file()
    and p.suffix.lower() in extensions
    and not any(part.lower() == "por clasificar" for part in p.parts)
)

if not paths:
    raise ValueError("No images found. Check the folder path.")

# Illustrative starting thresholds — adjust after visual inspection.
MIN_SIDE = 224
DARK_THRESHOLD = 50
BRIGHT_THRESHOLD = 205
BLUR_THRESHOLD = 100

records = []

for path in tqdm(paths):
    row = {
        "filename": path.relative_to(folder).as_posix(),
        "path": str(path),
        "readable": False,
        "sha256": None,
        "width": np.nan,
        "height": np.nan,
        "format": None,
        "brightness": np.nan,
        "sharpness": np.nan,
        "small_image": False,
        "potentially_dark": False,
        "potentially_bright": False,
        "potentially_blurry": False,
        "error": ""
    }

    try:
        # Same hash means identical file contents.
        hasher = hashlib.sha256()
        with path.open("rb") as f:
            for chunk in iter(lambda: f.read(1024 * 1024), b""):
                hasher.update(chunk)
        row["sha256"] = hasher.hexdigest()

        with Image.open(path) as image:
            row["format"] = image.format
            image.verify()

        # Reopen after verify(), apply orientation, and decode pixels.
        with Image.open(path) as image:
            image = ImageOps.exif_transpose(image).convert("RGB")
            image.load()

            width, height = image.size

            # Normalize the longest side for quality measurements.
            scale = 512 / max(width, height)
            size = (
                max(1, round(width * scale)),
                max(1, round(height * scale))
            )
            gray = np.array(
                image.resize(size, Image.Resampling.LANCZOS).convert("L")
            )

        brightness = float(gray.mean())
        sharpness = float(cv2.Laplacian(gray, cv2.CV_64F).var())

        row.update({
            "readable": True,
            "width": width,
            "height": height,
            "brightness": brightness,
            "sharpness": sharpness,
            "small_image": min(width, height) < MIN_SIDE,
            "potentially_dark": brightness < DARK_THRESHOLD,
            "potentially_bright": brightness > BRIGHT_THRESHOLD,
            "potentially_blurry": sharpness < BLUR_THRESHOLD
        })

    except Exception as error:
        row["error"] = f"{type(error).__name__}: {error}"

    records.append(row)

report = pd.DataFrame(records)

# Mark every member of an exact-duplicate group.
report["exact_duplicate"] = (
    report["sha256"].notna()
    & report["sha256"].duplicated(keep=False)
)

flags = [
    "small_image",
    "potentially_dark",
    "potentially_bright",
    "potentially_blurry",
    "exact_duplicate"
]

report["needs_review"] = (
    ~report["readable"] | report[flags].any(axis=1)
)

display(report)

| Column | Meaning |
|---|---|
| `brightness` | Average grayscale intensity: 0 is black and 255 is white |
| `sharpness` | Edge variation; lower values can indicate blur |
| `small_image` | Shortest side is below your chosen minimum |
| `exact_duplicate` | File contents match another file exactly |
| `needs_review` | At least one check flagged the image |

In [ ]:
checks = {
    "Unreadable images": ~report["readable"],
    "Small images": report["small_image"],
    "Potentially dark": report["potentially_dark"],
    "Potentially bright": report["potentially_bright"],
    "Potentially blurry": report["potentially_blurry"],
    "Images in exact-duplicate groups": report["exact_duplicate"],
    "Images requiring review": report["needs_review"]
}

summary = pd.DataFrame([
    {
        "Check": name,
        "Images": int(mask.sum()),
        "Percentage": round(mask.mean() * 100, 2)
    }
    for name, mask in checks.items()
])

print(f"Total images inspected: {len(report)}")
display(summary)

Total images inspected: 284


                              Check  Images  Percentage
0                 Unreadable images       0        0.00
1                      Small images       0        0.00
2                  Potentially dark       0        0.00
3                Potentially bright       0        0.00
4                Potentially blurry       1        0.35
5  Images in exact-duplicate groups      28        9.86
6           Images requiring review      29       10.21

In [ ]:
import matplotlib.pyplot as plt

flagged = report[
    report["readable"] & report["needs_review"]
].head(12)

if flagged.empty:
    print("No readable images were flagged.")
else:
    fig, axes = plt.subplots(
        len(flagged), 1,
        figsize=(8, 4 * len(flagged)),
        squeeze=False
    )

    for ax, (_, row) in zip(axes.flat, flagged.iterrows()):
        with Image.open(row["path"]) as image:
            ax.imshow(ImageOps.exif_transpose(image).convert("RGB"))

        reasons = [flag for flag in flags if row[flag]]
        ax.set_title(
            row["filename"] + "\n" + ", ".join(reasons),
            fontsize=9
        )
        ax.axis("off")

    plt.tight_layout()
    plt.show()